# Reaction Studio — Colab backend

Compute side only: nothing is edited here. This notebook mounts Drive, points at your capture and serves the editor — you cut and render in the browser tab.

Need the in-cell widgets GUI or manual render calls? Those live in `video_editor_colab.ipynb`. Docs: `colab_version/README.md`.

In [ ]:
BRANCH = "master"                                 # <- or any branch, e.g. a PR branch
REPO = "/content/VideoEditorTool"

!pip install -q numpy opencv-python mediapipe faster-whisper
import shutil, subprocess, sys
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                "https://github.com/RailRGO/VideoEditorTool", REPO], check=True)
sys.path.insert(0, f"{REPO}/colab_version")
from video_processor import ReactionVideoProcessor
from webapp.server import launch_webapp, stop_webapp

In [ ]:
INPUT  = "/content/drive/MyDrive/raw/recording_3840.mp4"   # file, or the folder holding your captures
OUTPUT = "/content/drive/MyDrive/reaction_output"

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
if Path(INPUT).is_dir():                       # a folder works too: first clip in it
    clips = [f for f in sorted(Path(INPUT).iterdir())
             if f.suffix.lower() in (".mp4", ".mkv", ".mov", ".webm", ".m4v", ".avi")]
    if not clips:
        raise SystemExit(f"no video files in {INPUT} (mp4/mkv/mov/webm)")
    INPUT = str(clips[0])
    print("using", INPUT)

proc = ReactionVideoProcessor(INPUT, output_dir=OUTPUT)

In [ ]:
HOSTED_EDITOR = "https://reaction-studio.onrender.com"   # <- your deployed UI (render.yaml). "" = built-in UI

web = launch_webapp(proc)                        # server + tunnel; keeps running in the background
if HOSTED_EDITOR and web.get("public"):
    from urllib.parse import quote
    print("one click ->",
          f"{HOSTED_EDITOR.rstrip('/')}/?backend={quote(web['public'], safe='')}")

In [ ]:
if "web" in globals():
    st = web["app"].state()
    print(st["info"]["path"], f'{st["info"]["duration"]:.0f}s',
          "proxy ready" if st["proxy"]["ready"] else f'proxy {st["proxy"]["progress"]:.0%}')
    print("renders ->", OUTPUT)

# re-run the cell above  -> fresh tunnel, your edit/state kept          ·  stop the server + tunnel:  stop_webapp()
# render without a browser:  proc.run_patron_version(intro_range=(0, 45), outro_range=(1250, 1290))
# in-cell GUI, no tunnel needed:  from editor_gui import launch_editor; launch_editor(proc)